In [8]:
# CELL 1: Install all required packages
!pip install -q --upgrade pip
!pip install -q transformers sentence-transformers faiss-cpu langchain huggingface_hub bitsandbytes

print("All packages installed! Please restart the runtime (Runtime > Restart runtime), then continue with Cell 2.")

All packages installed! Please restart the runtime (Runtime > Restart runtime), then continue with Cell 2.


In [9]:
# CELL 2: Login to HuggingFace (required for Llama models)
from huggingface_hub import login
login(token="")  # Replace with your HuggingFace token
print("Logged into HuggingFace")

Logged into HuggingFace


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("Libraries imported!")

# Load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded!")

# Use a small, fast chat model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

chat_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map=device
)
tokenizer.pad_token = tokenizer.eos_token
print(f"Chat model loaded: {model_name} on {device}")

Libraries imported!
Embedding model loaded!


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Chat model loaded: TinyLlama/TinyLlama-1.1B-Chat-v1.0 on cuda


In [11]:
# CELL 4: Test models and create MarcyAssistant

# Test embedding
embeddings = embedding_model.encode(["test"])
print(f"Embedding works! Shape: {embeddings.shape}")

# Test chat model
prompt = "What is 2 + 2?"
inputs = tokenizer(prompt, return_tensors="pt").to(chat_model.device)
outputs = chat_model.generate(**inputs, max_new_tokens=32, pad_token_id=tokenizer.eos_token_id)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Chat model response: {response[:100]}")

# MarcyAssistant class
class MarcyAssistant:
    def __init__(self, embedding_model, chat_model, tokenizer):
        self.embedding_model = embedding_model
        self.chat_model = chat_model
        self.tokenizer = tokenizer
        self.knowledge_base = []
        self.faiss_index = None

    def embed_text(self, text):
        return self.embedding_model.encode([text])[0]

    def generate_response(self, prompt, max_length=200):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.chat_model.device)
        outputs = self.chat_model.generate(
            **inputs,
            max_new_tokens=max_length,
            pad_token_id=self.tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.7
        )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

marcy = MarcyAssistant(embedding_model, chat_model, tokenizer)
print("STEP 1 COMPLETE! Marcy Assistant ready for Step 2!")

Embedding works! Shape: (1, 384)
Chat model response: What is 2 + 2?
STEP 1 COMPLETE! Marcy Assistant ready for Step 2!


In [12]:
from google.colab import files
uploaded = files.upload()  # Select your questions.csv file

Saving questions.csv to questions.csv


In [13]:
import pandas as pd

df = pd.read_csv('questions.csv')
print(df.head())  # Preview the first few rows

# Convert to list of dicts for embedding
lessons = df.to_dict(orient='records')
print(f"Loaded {len(lessons)} questions.")
print("Sample:", lessons[0])

   id                  question answer    category difficulty  \
0   1            What is 7 + 5?     12  Arithmetic       Easy   
1   2            What is 9 - 4?      5  Arithmetic       Easy   
2   3            What is 6 × 3?     18  Arithmetic       Easy   
3   4  Solve for x: 2x + 5 = 11      3     Algebra     Medium   
4   5  Solve for y: 3y - 4 = 11      5     Algebra     Medium   

                                         explanation  
0                   Add the two numbers: 7 + 5 = 12.  
1                        Subtract 4 from 9 to get 5.  
2  Multiplication means repeated addition: 6 × 3 ...  
3  Subtract 5 from both sides to get 2x = 6, then...  
4  Add 4 to both sides to get 3y = 15, then divid...  
Loaded 72 questions.
Sample: {'id': 1, 'question': 'What is 7 + 5?', 'answer': '12', 'category': 'Arithmetic', 'difficulty': 'Easy', 'explanation': 'Add the two numbers: 7 + 5 = 12.'}


In [14]:
# Use the 'question' field for embeddings
lesson_texts = [lesson['question'] for lesson in lessons]
lesson_embeddings = embedding_model.encode(lesson_texts, show_progress_bar=True)

import faiss
import numpy as np

dimension = lesson_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(np.array(lesson_embeddings))

print(f"FAISS index created with {faiss_index.ntotal} questions.")

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

FAISS index created with 72 questions.


In [15]:
marcy.knowledge_base = lessons
marcy.faiss_index = faiss_index
print("Knowledge base and FAISS index stored in MarcyAssistant.")

Knowledge base and FAISS index stored in MarcyAssistant.


In [16]:
# CELL: Retrieve most relevant question(s) from the knowledge base and print safely

def retrieve_relevant_questions(query, k=3):
    # Embed the query
    query_embedding = embedding_model.encode([query])
    # Search FAISS index
    D, I = marcy.faiss_index.search(np.array(query_embedding), k)
    # Get top-k results
    results = [marcy.knowledge_base[idx] for idx in I[0]]
    return results

# Example usage:
query = "How do you solve 2x + 5 = 11?"
top_questions = retrieve_relevant_questions(query, k=3)
for i, q in enumerate(top_questions, 1):
    print(f"Result {i}:")
    print("Question:", q.get('question', ''))
    print("Answer:", q.get('answer', ''))
    print("Topic:", q.get('topic', ''))
    print("Difficulty:", q.get('difficulty', ''))
    print("Explanation:", q.get('explanation', ''))
    print("-" * 40)

Result 1:
Question: Solve for x: 2x + 5 = 11
Answer: 3
Topic: 
Difficulty: Medium
Explanation: Subtract 5 from both sides to get 2x = 6, then divide by 2.
----------------------------------------
Result 2:
Question: Solve: 2x + 3 = 9
Answer: 3
Topic: 
Difficulty: Medium
Explanation: Subtract 3, then divide by 2.
----------------------------------------
Result 3:
Question: Solve: 2x^2 + 3x - 5 = 0
Answer: 1 or -5/2
Topic: 
Difficulty: Hard
Explanation: Factor or use quadratic formula.
----------------------------------------


In [17]:
# Example usage:
user_query = "How do I solve 2x + 5 = 11?"

def generate_step_by_step_answer(user_query, k=1):
    # Retrieve the most relevant question(s)
    retrieved = retrieve_relevant_questions(user_query, k=k)
    # Build the prompt with only the top example
    prompt = (
        f"Here is a similar math problem and its step-by-step solution:\n"
        f"Q: {retrieved[0]['question']}\nA: {retrieved[0]['answer']}\n\n"
        f"Now, answer this question step by step:\nQ: {user_query}\nA:"
    )
    # Generate the answer with a shorter max_length
    answer = marcy.generate_response(prompt, max_length=64)
    return answer

response = generate_step_by_step_answer(user_query)
print("LLM Step-by-Step Answer:\n", response)

LLM Step-by-Step Answer:
 Here is a similar math problem and its step-by-step solution:
Q: Solve for x: 2x + 5 = 11
A: 3

Now, answer this question step by step:
Q: How do I solve 2x + 5 = 11?
A: 

Step 1: Identify the variable(s) to be solved.

In this problem, the variable(s) to be solved are x.

Step 2: Find the value of the variable(s) using the given expression.

In this problem, the value of
